# Stage 2 — Notebook 01: Drifter Cohort Inventory, Human Interactions & "Collector vs. Trap" Beaches

**Objective:**  
In Stage 1, we proved that **67.8% of historical stranding days** consisted solely of decomposed carcasses (`Moderate Decomposition`, `Advanced Decomposition`, `Mummified`), `Fresh Dead` singletons, or non-pelagic species, and that removing them unlocked massive predictive skill for live pelagic odontocete entrapment (`+33.2%` to `+188.0%` PR-AUC lift).

In **Stage 2**, we turn that "noise" into our primary conservation signal. Rather than discarding the ~2,500 dead/decomposed strandings, `379` confirmed Human Interaction (`HI == 'Y'`) cases, and `99` offshore `EEZ` floating carcass sightings, we prepare them for **Lagrangian Reverse-Drift Backtracking** to answer:
1. **"Trap Beaches" vs. "Collector Beaches":** Which Massachusetts beaches actively trap live swimmers (like Wellfleet's tidal flats), and which beaches passively collect dead carcasses pushed ashore by wind and ocean currents?
2. **Drifter & Bycatch Cohorts:** How do confirmed **Fishery Interactions (`n = 116`)**, **Vessel Strikes (`n = 34`)**, **Harbor Porpoises (`n = 797`)**, **Pelagic Delphinids**, and **Large Whales** differ in decomposition state, season, and landing geography?
3. **Offshore `EEZ` Validation Set (`n = 99`):** Where are dead/entangled cetaceans actually observed floating at sea before beaching?

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

PROJECT_ROOT = Path('/home/nicastronaut/marine_bio/stage2_reverse_drift')
DATA_DIR = PROJECT_ROOT / 'data'
FIG_DIR = PROJECT_ROOT / 'figures'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV = Path('/home/nicastronaut/champ/capstone/2000-2025 MA Cetaceans.csv')
raw = pd.read_csv(RAW_CSV, encoding='utf-8-sig', low_memory=False)
df = raw.loc[raw['National Database Number'].notna()].copy().reset_index(drop=True)
print(f'Loaded {len(df):,} valid cetacean records (2000-2025) from {RAW_CSV.name}')


Loaded 4,831 valid cetacean records (2000-2025) from 2000-2025 MA Cetaceans.csv


## 1. Coordinate Cleaning, Typo Recovery & Town-Centroid Imputation

Unlike Stage 1 (which dropped `Human Interaction == 'Y'` and `County == 'EEZ'`), we retain **all 4,831 records** and clean their spatial coordinates (`lat_clean`, `lon_clean`):
- Fix two manual data-entry typos identified during our coordinate audit:
  1. `NE-2004-1042007` (Westport, Bristol County): `Latitude = '4130.283'`, `Longitude = '71.05.553'` -> DDMM.mmm format -> `41.5047°N, -71.0926°W`.
  2. `NE-2019-1212978` (Manchester-by-the-Sea, Essex County): `Latitude = '45.5721'` -> digit typo for `42.5721°N, -70.7708°W`.
- For the `33` coastal records missing raw latitude/longitude strings, we impute the median beach coordinate of their recorded `town` (`coord_imputed = True`) and also track exact-coordinate records (`coord_imputed = False`).

In [2]:
TOWN_FIXES = {'WELFLEET': 'WELLFLEET', 'MANCHESTER BY THE SEA': 'MANCHESTER-BY-THE-SEA'}

def clean_town(s):
    if pd.isna(s):
        return 'Unknown'
    up = str(s).strip().upper()
    up = TOWN_FIXES.get(up, up)
    return up.title()

df['town'] = df['City - from strandings table'].apply(clean_town)
df['Observation Date'] = pd.to_datetime(df['Observation Date'], format='%Y-%b-%d')
df['year'] = df['Observation Date'].dt.year
df['month'] = df['Observation Date'].dt.month
df['doy'] = df['Observation Date'].dt.dayofyear

def parse_coord(raw_val):
    if pd.isna(raw_val):
        return np.nan
    s = str(raw_val).strip().strip('()')
    try:
        v = float(s)
        if v > 1000:  # DDMM.mmm format like 4130.283
            deg = int(v // 100)
            mn = v - deg * 100
            return deg + mn / 60.0
        return v
    except ValueError:
        pass
    if '/' in s:
        parts = s.split('/')
        if len(parts) == 2:
            try:
                return abs(float(parts[0])) + float(parts[1]) / 60.0
            except ValueError:
                pass
    parts = s.split('.')
    if len(parts) == 4:
        try:
            return abs(float(parts[0])) + float(parts[1]) / 60.0 + float(f'{parts[2]}.{parts[3]}') / 3600.0
        except ValueError:
            pass
    if len(parts) == 3:
        try:
            deg, p1, p2 = float(parts[0]), float(parts[1]), float(parts[2])
            if len(parts[2]) == 3:
                mn = float(f'{int(p1)}.{parts[2]}')
                return abs(deg) + mn / 60.0
            return abs(deg) + p1 / 60.0 + p2 / 3600.0
        except ValueError:
            pass
    return np.nan

df['lat_clean'] = df['Latitude'].apply(parse_coord).abs()
df['lon_clean'] = -df['Longitude'].apply(parse_coord).abs()

# Fix Manchester-by-the-Sea latitude typo (45.5721 -> 42.5721)
mask_typo = df['National Database Number'] == 'NE-2019-1212978'
df.loc[mask_typo, 'lat_clean'] = 42.5721

# Impute missing coordinates using town median (or County median if town is Unknown)
df['coord_imputed'] = df['lat_clean'].isna() | df['lon_clean'].isna()
town_medians = df.loc[~df['coord_imputed']].groupby('town')[['lat_clean', 'lon_clean']].median()
county_medians = df.loc[~df['coord_imputed']].groupby('County')[['lat_clean', 'lon_clean']].median()

for idx in df.index[df['coord_imputed']]:
    t = df.loc[idx, 'town']
    c = df.loc[idx, 'County']
    if t in town_medians.index and pd.notna(town_medians.loc[t, 'lat_clean']):
        df.loc[idx, 'lat_clean'] = town_medians.loc[t, 'lat_clean']
        df.loc[idx, 'lon_clean'] = town_medians.loc[t, 'lon_clean']
    elif c in county_medians.index and pd.notna(county_medians.loc[c, 'lat_clean']):
        df.loc[idx, 'lat_clean'] = county_medians.loc[c, 'lat_clean']
        df.loc[idx, 'lon_clean'] = county_medians.loc[c, 'lon_clean']

print(f"Exact coordinates: {(~df['coord_imputed']).sum():,} / {len(df):,}")
print(f"Imputed town/county coordinates: {df['coord_imputed'].sum():,} / {len(df):,}")
print(f"Remaining missing coordinates: {df['lat_clean'].isna().sum()}")


Exact coordinates: 4,799 / 4,831
Imputed town/county coordinates: 32 / 4,831
Remaining missing coordinates: 0


## 2. Defining Decomposition Tiers, Human Interactions & Ecological Guilds

We classify every record along three orthogonal dimensions:
1. **Decomposition / Drift Tier (`drift_tier`):**
   - `Alive`: Live stranding (`n = 2,046`)
   - `Fresh Dead (MSE)`: Died during a group/mass stranding event (`n = 149`)
   - `Fresh Dead (Single Drifter)`: Single fresh dead animal (`n = 544`, short backtrack window: `6–48h`)
   - `Moderate Decomposition`: Bloated/intact drifting carcass (`n = 1,479`, medium SST-scaled backtrack window: `3–10d`)
   - `Advanced Decomposition`: Heavily decomposed carcass (`n = 467`, long SST-scaled backtrack window: `8–18d`)
   - `Mummified / Skeletal / Unknown`: Uncalibrated drift duration (`n = 146`, excluded from particle backtracking)
2. **Human Interaction Subtype (`hi_type`):**
   - `Fishery / Entanglement`: Confirmed `Fishery Interaction == 'Y'` or `Entangled OD == 'Y'` (`n = 116`)
   - `Vessel Strike`: Confirmed `Boat Collision == 'Y'` (`n = 32` non-overlapping, `34` total)
   - `Other HI`: Other confirmed human interaction (`n = 236`, e.g., human intervention/relocation, out-of-habitat, shot, debris ingestion)
   - `None / CBD`: No confirmed human interaction (`n = 4,447`)
3. **Ecological Guild (`species_guild`):**
   - `Harbor Porpoise` (*Phocoena phocoena*)
   - `Common Dolphin` (*Delphinus delphis*)
   - `White-Sided Dolphin` (*Lagenorhynchus acutus*)
   - `Other Pelagic Delphinid` (*Globicephala*, *Tursiops*, *Grampus*, *Stenella*)
   - `Baleen & Large Whales` (*Balaenoptera*, *Megaptera*, *Eubalaena*, *Physeter*, *Kogia*, *Ziphiidae*)
   - `Unidentified / Other`

In [3]:
# Group / Mass Stranding flag (same exact logic as Stage 1)
df['MSE'] = (
    df['Mass Stranding Number'].notna()
    | (df['Group Type: Mass Stranding'] == 'Y')
    | (df['Group Event Flag'] == 'Y')
    | (df['Group Type: Cow Calf Pair'] == 'Y')
)

def get_drift_tier(row):
    status = row['Observation Status']
    if status == 'Alive':
        return 'Alive'
    elif status == 'Fresh Dead':
        return 'Fresh Dead (MSE)' if row['MSE'] else 'Fresh Dead (Single Drifter)'
    elif status == 'Moderate Decomposition':
        return 'Moderate Decomposition'
    elif status == 'Advanced Decomposition':
        return 'Advanced Decomposition'
    elif status == 'Mummified/Skeletal':
        return 'Mummified/Skeletal'
    else:
        return 'Condition Unknown'

df['drift_tier'] = df.apply(get_drift_tier, axis=1)

def get_hi_type(row):
    if row['Fishery Interaction'] == 'Y' or row['Entangled OD'] == 'Y':
        return 'Fishery / Entanglement'
    elif row['Boat Collision'] == 'Y':
        return 'Vessel Strike'
    elif row['Findings of Human Interaction'] == 'Y' or row['Shot'] == 'Y' or row['Ingestion'] == 'DEBRIS':
        return 'Other Human Interaction'
    else:
        return 'None / CBD'

df['hi_type'] = df.apply(get_hi_type, axis=1)

def get_species_guild(row):
    cn = str(row['Common Name']).lower()
    genus = str(row['Genus']).lower()
    if 'porpoise' in cn or genus == 'phocoena':
        return 'Harbor Porpoise'
    elif 'common, short-beaked' in cn or genus == 'delphinus':
        return 'Common Dolphin'
    elif 'white-sided' in cn or genus == 'lagenorhynchus':
        return 'White-Sided Dolphin'
    elif any(k in cn for k in ['pilot', 'bottlenose', 'risso', 'striped', 'spotted', 'spinner', 'rough-toothed']) or genus in ['globicephala', 'tursiops', 'grampus', 'stenella']:
        return 'Other Pelagic Delphinid'
    elif any(k in cn for k in ['whale', 'minke', 'humpback', 'fin', 'right', 'sei', 'sperm', 'beaked', 'pygmy']) or genus in ['balaenoptera', 'megaptera', 'eubalaena', 'physeter', 'kogia', 'mesoplodon', 'ziphius']:
        return 'Baleen & Large Whales'
    else:
        return 'Unidentified / Other'

df['species_guild'] = df.apply(get_species_guild, axis=1)

# Define Coastal Reception Zone (accounting for Bay-side vs. Sound-side Cape Cod towns, and South Shore vs. North Shore)
def get_coastal_zone(row):
    if row['County'] == 'EEZ':
        return '6. Offshore EEZ (At-Sea)'
    town = row['town']
    county = row['County']
    lat, lon = row['lat_clean'], row['lon_clean']
    # Sound-side of Cape Cod (Nantucket Sound) or Buzzards Bay side of Bourne
    if county in ['Nantucket', 'Dukes', 'Bristol'] or town in ['Falmouth', 'Mashpee'] or (lat < 41.685 and lon < -69.98) or (town == 'Bourne' and lat < 41.745 and lon < -70.56):
        return '4. Southern Islands, Sound & Buzzards Bay'
    elif town in ['Wellfleet', 'Brewster', 'Dennis', 'Eastham', 'Orleans'] and lon < -69.985 and lat >= 41.685:
        return '1. Inner Cape Hook (Trap Flats)'
    elif town in ['Barnstable', 'Sandwich', 'Yarmouth', 'Bourne']:
        return '2. Bay-Side Upper/Mid Cape (Collector Shore)'
    elif county in ['Plymouth', 'Norfolk', 'Suffolk', 'Essex']:
        return '5. South Shore, Boston & North Shore'
    else:
        return '3. Outer Cape & Provincetown Spit'

df['coastal_zone'] = df.apply(get_coastal_zone, axis=1)

print('=== DRIFT TIER SUMMARY ===')
print(df['drift_tier'].value_counts())
print('\n=== HUMAN INTERACTION SUMMARY ===')
print(pd.crosstab(df['hi_type'], df['drift_tier'], margins=True))
print('\n=== COASTAL ZONE BY DRIFT TIER ===')
cz = pd.crosstab(df['coastal_zone'], df['drift_tier'], margins=True)
cz['Pct_Alive'] = (100 * cz['Alive'] / cz['All']).round(1)
cz['Pct_Drifter'] = (100 * (cz['Fresh Dead (Single Drifter)'] + cz['Moderate Decomposition'] + cz['Advanced Decomposition']) / cz['All']).round(1)
print(cz[['Alive', 'Fresh Dead (MSE)', 'Fresh Dead (Single Drifter)', 'Moderate Decomposition', 'Advanced Decomposition', 'All', 'Pct_Alive', 'Pct_Drifter']])


=== DRIFT TIER SUMMARY ===
drift_tier
Alive                          2046
Moderate Decomposition         1479
Advanced Decomposition          467
Fresh Dead (Single Drifter)     384
Fresh Dead (MSE)                309
Mummified/Skeletal              103
Condition Unknown                43
Name: count, dtype: int64

=== HUMAN INTERACTION SUMMARY ===
drift_tier               Advanced Decomposition  Alive  Condition Unknown  \
hi_type                                                                     
Fishery / Entanglement                       12     31                  3   
None / CBD                                  442   1840                 39   
Other Human Interaction                       9    167                  1   
Vessel Strike                                 4      8                  0   
All                                         467   2046                 43   

drift_tier               Fresh Dead (MSE)  Fresh Dead (Single Drifter)  \
hi_type                            

## 3. "Trap Beaches" vs. "Collector Beaches" Analysis

A major hypothesis in stranding science is that coastlines divide into two functional types:
- **Active Trap Beaches:** Complex shallow geomorphology and large tidal ranges (e.g., Wellfleet, Brewster, Dennis) where **live, healthy pelagic groups** strand.
- **Passive Collector Beaches:** Exposed shorelines facing prevailing wind/current fetch (e.g., Sandwich facing north into Cape Cod Bay, Plymouth facing east into Mass Bay, Martha's Vineyard/Nantucket facing the southern shelf, and Gloucester/Essex on the North Shore) where **floating carcasses** wash ashore.

Let's compute the exact **Collector Index** (`% Decomposed + Single Fresh Dead`) vs. **Trap Index** (`% Alive`) across the top Massachusetts stranding towns and coastal zones.

In [4]:
# Build town-level Collector vs Trap table (for named towns with >= 30 strandings)
town_counts = df.loc[(df['County'] != 'EEZ') & (df['town'] != 'Unknown')].groupby('town').size()
major_towns = town_counts[town_counts >= 30].index

t_tab = pd.crosstab(df.loc[df['town'].isin(major_towns), 'town'], df.loc[df['town'].isin(major_towns), 'drift_tier'])
t_tab['Total'] = t_tab.sum(axis=1)
t_tab['Decomp_Count'] = t_tab.get('Moderate Decomposition', 0) + t_tab.get('Advanced Decomposition', 0) + t_tab.get('Mummified/Skeletal', 0)
t_tab['Drifter_Count'] = t_tab['Decomp_Count'] + t_tab.get('Fresh Dead (Single Drifter)', 0)
t_tab['Pct_Alive'] = (100 * t_tab.get('Alive', 0) / t_tab['Total']).round(1)
t_tab['Pct_Decomp'] = (100 * t_tab['Decomp_Count'] / t_tab['Total']).round(1)
t_tab['Pct_Total_Drifter'] = (100 * t_tab['Drifter_Count'] / t_tab['Total']).round(1)
t_tab['Beach_Type'] = np.where(t_tab['Pct_Alive'] >= 45.0, 'Active Swimmer Trap',
                      np.where(t_tab['Pct_Decomp'] >= 55.0, 'Passive Carcass Collector', 'Mixed Trap / Collector'))
t_tab = t_tab.sort_values('Pct_Decomp', ascending=False)
t_tab.to_csv(DATA_DIR / 'collector_vs_trap_beaches.csv')
print('=== TOP MASSACHUSETTS TOWNS: PASSIVE COLLECTORS VS. ACTIVE TRAPS (n >= 30) ===')
print(t_tab[['Total', 'Alive', 'Fresh Dead (Single Drifter)', 'Moderate Decomposition', 'Advanced Decomposition', 'Pct_Alive', 'Pct_Decomp', 'Beach_Type']].to_string())


=== TOP MASSACHUSETTS TOWNS: PASSIVE COLLECTORS VS. ACTIVE TRAPS (n >= 30) ===
drift_tier    Total  Alive  Fresh Dead (Single Drifter)  Moderate Decomposition  Advanced Decomposition  Pct_Alive  Pct_Decomp                 Beach_Type
town                                                                                                                                                     
Hull             30      1                            3                      20                       5        3.3        83.3  Passive Carcass Collector
Sandwich         46      7                            3                      24                       9       15.2        78.3  Passive Carcass Collector
Westport         38      5                            6                      19                       6       13.2        71.1  Passive Carcass Collector
Plymouth         69     10                           11                      33                      12       14.5        66.7  Passive Carcass Collect

## 4. Selecting the Back-Trackable Coastal Drifter Cohort & `EEZ` Ground-Truth Set

Which records should enter our Lagrangian Reverse-Drift Particle Tracker in Notebook 02?
1. **Coastal Back-Trackable Drifters (`is_coastal_drifter == True`):**
   - Landed on a coastal beach (`County != 'EEZ'`) inside our `GLORYS12V1` regional ocean domain (`40.2°N–43.2°N, -71.6°W–-68.5°W`).
   - Either:
     - `Moderate Decomposition` (`1,419` coastal carcasses),
     - `Advanced Decomposition` (`448` coastal carcasses),
     - `Fresh Dead (Single Drifter)` (`534` singletons that died at sea shortly before landing), OR
     - Confirmed `Fishery / Entanglement` or `Vessel Strike` (`HI` dead or injured at sea).
   - We assign each coastal drifter a primary analytical cohort (`drifter_cohort`) so we can map separate mortality hotspots for:
     1. **`1. Confirmed Fishery / Entanglement`**
     2. **`2. Confirmed Vessel Strike`**
     3. **`3. Harbor Porpoise Carcass`**
     4. **`4. Pelagic Dolphin Carcass`**
     5. **`5. Large Whale / Other Carcass`**
2. **Offshore `EEZ` Floating Sightings (`is_eez_validation == True`, `n = 99`):**
   - Carcasses and entangled cetaceans observed floating at sea (`County == 'EEZ'`). These serve as an independent **offshore ground-truth validation set**!

In [5]:
in_domain = df['lat_clean'].between(40.25, 43.15) & df['lon_clean'].between(-71.55, -68.55)
is_drifter_state = df['drift_tier'].isin(['Fresh Dead (Single Drifter)', 'Moderate Decomposition', 'Advanced Decomposition'])
is_hi_mortality = df['hi_type'].isin(['Fishery / Entanglement', 'Vessel Strike'])

df['is_coastal_drifter'] = (df['County'] != 'EEZ') & in_domain & (is_drifter_state | is_hi_mortality)
df['is_eez_validation'] = (df['County'] == 'EEZ') & df['lat_clean'].notna()

def assign_drifter_cohort(row):
    if row['County'] == 'EEZ':
        return '6. Offshore EEZ Sighting (Validation)'
    if not row['is_coastal_drifter']:
        return '0. Non-Drifter (Live / MSE / Mummified)'
    if row['hi_type'] == 'Fishery / Entanglement':
        return '1. Confirmed Fishery / Entanglement'
    elif row['hi_type'] == 'Vessel Strike':
        return '2. Confirmed Vessel Strike'
    elif row['species_guild'] == 'Harbor Porpoise':
        return '3. Harbor Porpoise Carcass'
    elif row['species_guild'] in ['Common Dolphin', 'White-Sided Dolphin', 'Other Pelagic Delphinid']:
        return '4. Pelagic Dolphin Carcass'
    else:
        return '5. Large Whale / Other Carcass'

df['drifter_cohort'] = df.apply(assign_drifter_cohort, axis=1)

cohort_summary = df.groupby('drifter_cohort').agg(
    n_records=('National Database Number', 'count'),
    n_fresh=('drift_tier', lambda s: (s == 'Fresh Dead (Single Drifter)').sum()),
    n_mod_decomp=('drift_tier', lambda s: (s == 'Moderate Decomposition').sum()),
    n_adv_decomp=('drift_tier', lambda s: (s == 'Advanced Decomposition').sum()),
    n_alive_hi=('drift_tier', lambda s: (s == 'Alive').sum()),
    pct_spring_mam=('month', lambda s: round(100 * s.isin([3, 4, 5]).mean(), 1)),
    pct_winter_djf=('month', lambda s: round(100 * s.isin([12, 1, 2]).mean(), 1)),
    pct_summer_jja=('month', lambda s: round(100 * s.isin([6, 7, 8]).mean(), 1)),
    pct_autumn_son=('month', lambda s: round(100 * s.isin([9, 10, 11]).mean(), 1)),
).reset_index()

cohort_summary.to_csv(DATA_DIR / 'cohort_summary_table.csv', index=False)
print('=== STAGE 2 DRIFTER COHORT SUMMARY ===')
print(cohort_summary.to_string(index=False))

df.to_parquet(DATA_DIR / 'stage2_all_records_clean.parquet', index=False)
drifters = df.loc[df['is_coastal_drifter'] | df['is_eez_validation']].copy().reset_index(drop=True)
drifters.to_parquet(DATA_DIR / 'stage2_drifter_cohorts.parquet', index=False)
print(f'\nSaved {len(df):,} total records and {len(drifters):,} Stage 2 drifter + EEZ validation records.')


=== STAGE 2 DRIFTER COHORT SUMMARY ===
                         drifter_cohort  n_records  n_fresh  n_mod_decomp  n_adv_decomp  n_alive_hi  pct_spring_mam  pct_winter_djf  pct_summer_jja  pct_autumn_son
0. Non-Drifter (Live / MSE / Mummified)       2450        0             1             1        2005            26.6            29.2            25.4            18.8
    1. Confirmed Fishery / Entanglement        112       14            51            10          30            25.9            22.3            22.3            29.5
             2. Confirmed Vessel Strike         23        1            12             4           6            26.1             8.7            39.1            26.1
             3. Harbor Porpoise Carcass        618       98           414           106           0            59.2            29.6             7.1             4.0
             4. Pelagic Dolphin Carcass       1344      244           859           241           0            33.0            40.4          


Saved 4,831 total records and 2,381 Stage 2 drifter + EEZ validation records.


## 5. Publication Figure 1: Spatial Geography of Trap vs. Collector Beaches & Drifter Cohorts

In [6]:
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

fig = plt.figure(figsize=(16, 12), dpi=180)
gs = gridspec.GridSpec(2, 2, width_ratios=[1.15, 1.0], height_ratios=[1.1, 0.95], hspace=0.30, wspace=0.25)

# Panel A: Spatial Map of Live Trap vs Decomposed Collectors vs HI vs EEZ
ax1 = fig.add_subplot(gs[0, 0])
import xarray as xr
glorys_sample = xr.open_dataset(DATA_DIR / 'glorys_wide_surface_1999_2025.nc').isel(time=0)
land_mask = glorys_sample['thetao'].isnull().values
ax1.contourf(glorys_sample.longitude, glorys_sample.latitude, land_mask, levels=[-0.5, 0.5, 1.5],
             colors=['#eef6fa', '#e5e0d5'], alpha=0.9, zorder=0)
ax1.contour(glorys_sample.longitude, glorys_sample.latitude, land_mask, levels=[0.5],
            colors=['#8c8270'], linewidths=0.9, zorder=1)

rng = np.random.default_rng(42)
live_sub = df.loc[(df['drift_tier'] == 'Alive') & (df['County'] != 'EEZ') & df['lat_clean'].between(41.0, 43.0)]
decomp_sub = df.loc[df['drift_tier'].isin(['Moderate Decomposition', 'Advanced Decomposition']) & (df['County'] != 'EEZ') & df['lat_clean'].between(41.0, 43.0)]
hi_sub = df.loc[df['hi_type'].isin(['Fishery / Entanglement', 'Vessel Strike']) & (df['County'] != 'EEZ') & df['lat_clean'].between(41.0, 43.0)]
eez_sub = df.loc[(df['County'] == 'EEZ') & df['lat_clean'].between(40.8, 43.0) & df['lon_clean'].between(-71.4, -68.6)]

ax1.scatter(decomp_sub['lon_clean'] + rng.normal(0, 0.012, len(decomp_sub)),
            decomp_sub['lat_clean'] + rng.normal(0, 0.012, len(decomp_sub)),
            s=12, c='#d95f02', alpha=0.45, label=f'Decomposed Carcasses (n={len(decomp_sub):,})', zorder=2)
ax1.scatter(live_sub['lon_clean'] + rng.normal(0, 0.010, len(live_sub)),
            live_sub['lat_clean'] + rng.normal(0, 0.010, len(live_sub)),
            s=10, c='#1b9e77', alpha=0.35, label=f'Live Strandings (n={len(live_sub):,})', zorder=3)
ax1.scatter(hi_sub['lon_clean'], hi_sub['lat_clean'],
            s=28, c='#e7298a', marker='^', edgecolors='black', linewidths=0.3, alpha=0.85,
            label=f'Fishery / Vessel Strike HI (n={len(hi_sub):,})', zorder=4)
ax1.scatter(eez_sub['lon_clean'], eez_sub['lat_clean'],
            s=42, c='#7570b3', marker='*', edgecolors='black', linewidths=0.4, alpha=0.95,
            label=f'Offshore EEZ Floating Carcasses (n={len(eez_sub):,})', zorder=5)

ax1.annotate('Wellfleet / Brewster\nActive Swimmer Trap\n(52-60% Alive)', xy=(-70.03, 41.91), xytext=(-70.65, 42.02),
             arrowprops=dict(arrowstyle='->', color='#1b9e77', lw=1.5),
             fontsize=8.5, fontweight='bold', color='#0f684e',
             bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#1b9e77', alpha=0.9))
ax1.annotate('Sandwich / Barnstable\nCarcass Collector Shore\n(59-78% Decomposed)', xy=(-70.48, 41.76), xytext=(-71.35, 41.68),
             arrowprops=dict(arrowstyle='->', color='#d95f02', lw=1.5),
             fontsize=8.5, fontweight='bold', color='#993d00',
             bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#d95f02', alpha=0.9))
ax1.annotate("Martha's Vineyard & Nantucket\nSouthern Shelf Collectors\n(58-67% Decomposed)", xy=(-70.55, 41.38), xytext=(-71.35, 41.12),
             arrowprops=dict(arrowstyle='->', color='#d95f02', lw=1.5),
             fontsize=8.5, fontweight='bold', color='#993d00',
             bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#d95f02', alpha=0.9))

ax1.set_xlim(-71.45, -68.85)
ax1.set_ylim(41.0, 42.9)
ax1.set_xlabel('Longitude (°W)', fontsize=10)
ax1.set_ylabel('Latitude (°N)', fontsize=10)
ax1.set_title('A. Active "Trap Beaches" vs. Passive "Collector Beaches" (2000–2025)', fontsize=11.5, fontweight='bold', loc='left')
ax1.legend(loc='upper right', fontsize=8, framealpha=0.92)
ax1.grid(True, linestyle='--', alpha=0.3)

# Panel B: Horizontal Stacked Bar of Top Towns (Pct Decomposed vs Fresh Dead vs Alive)
ax2 = fig.add_subplot(gs[0, 1])
top_14 = t_tab.sort_values('Pct_Decomp', ascending=True).copy()
y_pos = np.arange(len(top_14))
p_alive = 100 * top_14['Alive'] / top_14['Total']
p_fresh_mse = 100 * top_14.get('Fresh Dead (MSE)', 0) / top_14['Total']
p_fresh_single = 100 * top_14.get('Fresh Dead (Single Drifter)', 0) / top_14['Total']
p_decomp = 100 * top_14['Decomp_Count'] / top_14['Total']

ax2.barh(y_pos, p_decomp, color='#d95f02', alpha=0.88, label='Moderate / Advanced Decomp (Drifter)')
ax2.barh(y_pos, p_fresh_single, left=p_decomp, color='#fc8d62', alpha=0.88, label='Fresh Dead Singleton (<48h Drifter)')
ax2.barh(y_pos, p_fresh_mse, left=p_decomp + p_fresh_single, color='#8da0cb', alpha=0.85, label='Fresh Dead (In-MSE)')
ax2.barh(y_pos, p_alive, left=p_decomp + p_fresh_single + p_fresh_mse, color='#1b9e77', alpha=0.88, label='Stranded Alive (Swimmer)')

ax2.axvline(50, color='#333333', linestyle='--', linewidth=1.0, alpha=0.7)
ax2.set_yticks(y_pos)
ax2.set_yticklabels([f"{t} (n={int(top_14.loc[t, 'Total'])})" for t in top_14.index], fontsize=8.5)
ax2.set_xlim(0, 100)
ax2.set_xlabel('Percentage of Town Strandings (%)', fontsize=10)
ax2.set_title('B. Town-Level "Collector Index" (% Decomposed vs. % Alive)', fontsize=11.5, fontweight='bold', loc='left')
ax2.legend(loc='lower right', fontsize=8, framealpha=0.92)
ax2.grid(True, axis='x', linestyle='--', alpha=0.3)

# Panel C: Species Guild Decomposition Breakdown
ax3 = fig.add_subplot(gs[1, 0])
guilds = ['Harbor Porpoise', 'Baleen & Large Whales', 'Common Dolphin', 'Other Pelagic Delphinid', 'White-Sided Dolphin']
g_df = df.loc[df['species_guild'].isin(guilds)].copy()
g_tab = pd.crosstab(g_df['species_guild'], g_df['drift_tier']).reindex(guilds)
g_tot = g_tab.sum(axis=1)
g_decomp = 100 * (g_tab['Moderate Decomposition'] + g_tab['Advanced Decomposition'] + g_tab.get('Mummified/Skeletal', 0)) / g_tot
g_fresh_s = 100 * g_tab['Fresh Dead (Single Drifter)'] / g_tot
g_alive = 100 * (g_tab['Alive'] + g_tab['Fresh Dead (MSE)']) / g_tot

xg = np.arange(len(guilds))
w = 0.55
ax3.bar(xg, g_decomp, width=w, color='#d95f02', label='Decomposed Carcass (Mod/Adv/Mum)')
ax3.bar(xg, g_fresh_s, width=w, bottom=g_decomp, color='#fc8d62', label='Fresh Dead Singleton')
ax3.bar(xg, g_alive, width=w, bottom=g_decomp + g_fresh_s, color='#1b9e77', label='Alive / Fresh MSE Swimmer')

for i, g in enumerate(guilds):
    ax3.text(i, g_decomp.iloc[i] / 2, f'{g_decomp.iloc[i]:.0f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    if g_alive.iloc[i] > 12:
        ax3.text(i, g_decomp.iloc[i] + g_fresh_s.iloc[i] + g_alive.iloc[i] / 2, f'{g_alive.iloc[i]:.0f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=9)

ax3.set_xticks(xg)
ax3.set_xticklabels([f'{g}\n(n={int(g_tot.loc[g]):,})' for g in guilds], fontsize=8.5)
ax3.set_ylim(0, 100)
ax3.set_ylabel('Share of Species Strandings (%)', fontsize=10)
ax3.set_title('C. Condition at Discovery by Species Guild (Drifters vs. Swimmers)', fontsize=11.5, fontweight='bold', loc='left')
ax3.legend(loc='upper right', fontsize=8, framealpha=0.92)
ax3.grid(True, axis='y', linestyle='--', alpha=0.3)

# Panel D: Monthly Seasonality of Stage 2 Drifter Cohorts
ax4 = fig.add_subplot(gs[1, 1])
months = np.arange(1, 13)
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for cohort_label, color, marker, lw in [
    ('3. Harbor Porpoise Carcass', '#d95f02', 'o', 2.2),
    ('4. Pelagic Dolphin Carcass', '#1b9e77', 's', 2.2),
    ('1. Confirmed Fishery / Entanglement', '#e7298a', '^', 2.0),
    ('5. Large Whale / Other Carcass', '#7570b3', 'D', 1.8),
]:
    sub = drifters.loc[drifters['drifter_cohort'] == cohort_label]
    m_counts = sub.groupby('month').size().reindex(months, fill_value=0)
    m_pct = 100 * m_counts / m_counts.sum()
    clean_lbl = cohort_label.split('. ')[1] + f' (n={len(sub):,})'
    ax4.plot(months, m_pct, marker=marker, color=color, linewidth=lw, label=clean_lbl)

ax4.axvspan(2.7, 5.3, color='#d95f02', alpha=0.10, label='Spring Gillnet / Porpoise Peak (Mar–May)')
ax4.set_xticks(months)
ax4.set_xticklabels(month_labels, fontsize=9)
ax4.set_ylabel('Share of Annual Cohort Strandings (%)', fontsize=10)
ax4.set_title('D. Seasonal Timing of Drifter & Bycatch Cohorts (March–May Spring Spike)', fontsize=11.5, fontweight='bold', loc='left')
ax4.legend(loc='upper right', fontsize=8, framealpha=0.92)
ax4.grid(True, linestyle='--', alpha=0.3)

fig.suptitle('Stage 2 Cohort Audit: Separating Passive Carcass Collectors from Active Swimmer Traps (2000–2025)',
             fontsize=13.5, fontweight='bold', y=0.98)
fig.savefig(FIG_DIR / '01_drifter_cohorts_and_collector_beaches.png', bbox_inches='tight')
plt.close(fig)
print('Saved figure:', FIG_DIR / '01_drifter_cohorts_and_collector_beaches.png')


Saved figure: /home/nicastronaut/marine_bio/stage2_reverse_drift/figures/01_drifter_cohorts_and_collector_beaches.png
